In [ ]:
import inspect
import math
import random


def bits_per_variable(limits, precision):
    minimum, maximum = limits
    values = (maximum - minimum) / precision + 1
    return math.ceil(math.log2(values))

def bits_per_individual(bits):
    return ''.join(
            random.choice('01')
            for _ in range(sum(bits))
        )
    
def binary_to_decimal(bin, minimum, maximum):
    decimal = int(bin, 2)

    max_decimal = 2 ** len(bin) - 1

    return minimum + (decimal / max_decimal) * (maximum - minimum)

def decode(individual, bits, limits):
    values = []
    start = 0
    
    for n_bits, limit in zip(bits, limits):
        minimum, maximum = limit
        end = start + n_bits
        
        gene = individual[start:end]
        value = binary_to_decimal(gene, minimum, maximum)
        values.append(value)
        
        start = end
        
    return values

    
def fitness(individual, fun, bits, limits):
    values = decode(individual, bits, limits)
    return fun(*values)

def tournament_selection(pop, k):
    tournament = random.sample(range(len(pop)), k)
    winner = min(tournament)
    return pop[winner]

def mutate(individual, mutation_prob):
    mutated = ""

    for bit in individual:

        if random.random() < mutation_prob:
            mutated += '1' if bit == '0' else '0'
        else:
            mutated += bit

    return mutated

def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)

    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]

    return child1, child2



def gen_search(fun, obj, limits, precision, population_size = 100, 
               generations = 100, mutation_prob = 0.01, crossover_prob = 0.7, tournament_size = 3, logs = True):

    gens = list(inspect.signature(fun).parameters.keys())

    # Si solo se proporciona un límite,
    # utilizarlo para todas las variables
    if len(limits) == 1:
        limits = limits * len(gens)

    # Si hay varios límites, debe haber uno por variable
    elif len(limits) != len(gens):
        raise ValueError(
            f"Expected {len(gens)} limits, but received {len(limits)}."
        )

    for limit in limits:

        if len(limit) != 2:
            raise ValueError(
                "Each limit must contain exactly two values: (min, max)."
            )

        minimum, maximum = limit

        if minimum >= maximum:
            raise ValueError(
                f"Invalid limit {limit}: minimum ({minimum}) must be less than maximum ({maximum})."
            )
            
    bits = [
        bits_per_variable(limit, precision)
        for limit in limits
    ]
        

    population = [
        bits_per_individual(bits)
        for _ in range(population_size)
    ]
    
    if logs:
        print(f"Variables and their corresponding bits: {gens}")

        print(f"Initial population: {population}")
                    
        print(f"Bits per variable: {bits}")
    
    
    for generation in range(generations):

        # Ordenar 
        population.sort(
            key=lambda individuo:
            fitness(individuo, fun, bits, limits),
            reverse=(obj == "max")
        )
        
        best = population[0]
        
        values = decode(best, bits, limits)
        value = fun(*values)
        
        if logs:
        
            variables_text = " | ".join(
                f"{nombre} = {valor:.5f}"
                for nombre, valor in zip(gens, values)
            )

            print(
                f"Generation: {generation:3} | "
                f"{variables_text} | "
                f"f = {value:.8f}"
            )
        
        new_population = [best]
        
        while len(new_population) < population_size:
            
            parent1 = tournament_selection(population, tournament_size)
            parent2 = tournament_selection(population, tournament_size)

            if random.random() < crossover_prob:
                child1, child2 = crossover(parent1, parent2)
            else:
                child1, child2 = parent1, parent2

            child1 = mutate(child1, mutation_prob)
            child2 = mutate(child2, mutation_prob)

            new_population.append(child1)

            if len(new_population) < population_size:
                new_population.append(child2)
        population = new_population

    return decode(best, bits, limits), fun(*decode(best, bits, limits))

([-10.908563755112006, 1.1899645952875106], 0.1833471055508769)

In [ ]:
f = lambda x, y: (100 * math.sqrt(abs(y - 0.01 * x**2)) + 0.01 * abs(x + 10))

gen_search(
    fun=f,
    obj="min",
    limits=[(-15, -5), (-3, 3)],
    precision=0.001,
    population_size=1000,
    generations=500,
    logs=False,
    mutation_prob=0.01,
    crossover_prob=0.9,
    tournament_size=3
)